In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_regression

## Data Setup

In [2]:
# target variables: 'proctor_mdd_g_cm3', 'proctor_owc_pct'
df = pd.read_csv('./data/train.csv')
features = df.columns.drop(['id', 'proctor_mdd_g_cm3', 'proctor_owc_pct'])

df['clay'] = df['psd_passing_at_0_002mm_pct']
df['silt'] = df['psd_passing_at_0_063mm_pct'] - df['psd_passing_at_0_002mm_pct']
df['sand'] = df['psd_passing_at_2mm_pct'] - df['psd_passing_at_0_063mm_pct']
df['gravel'] = 100 - df['psd_passing_at_2mm_pct']
df['fine-grained'] = (df['clay'] + df['silt'] > 12)
df['ip'] = df['atterberg_liquid_limit_pct'] - df['atterberg_plastic_limit_pct']

## Correlation

In [9]:
corr_mat = df[features].corr().round(5)
corr_mat

,psd_size_at_d10_mm,psd_size_at_d20_mm,psd_size_at_d30_mm,psd_size_at_d40_mm,psd_size_at_d50_mm,psd_size_at_d60_mm,psd_size_at_d70_mm,psd_size_at_d80_mm,psd_size_at_d90_mm,psd_size_at_d95_mm,...,psd_passing_at_0_002mm_pct,psd_passing_at_0_063mm_pct,psd_passing_at_2mm_pct,proctor_diam_mm,grain_density_g_cm3,hyd_cond_kf_m_s,hyd_cond_hyd_gradient,atterberg_liquid_limit_pct,atterberg_plastic_limit_pct,loss_on_ignition_pct
psd_size_at_d10_mm,1.00000,0.78159,0.68758,0.65825,0.62035,0.55870,0.49860,0.49457,0.51030,0.49970,...,-0.49730,-0.66520,-0.63143,0.44493,-0.12161,0.49271,-0.27113,-0.41985,-0.30393,-0.62057
psd_size_at_d20_mm,0.78159,1.00000,0.97603,0.88344,0.77370,0.66269,0.56438,0.51681,0.47931,0.44448,...,-0.35746,-0.46242,-0.61235,0.41755,-0.04299,0.40563,-0.34144,-0.51428,-0.46817,-0.68912
psd_size_at_d30_mm,0.68758,0.97603,1.00000,0.94286,0.83413,0.71345,0.60496,0.54407,0.48903,0.44694,...,-0.30353,-0.38744,-0.61342,0.41800,-0.02817,0.29399,-0.35328,-0.53771,-0.55597,-0.64485
psd_size_at_d40_mm,0.65825,0.88344,0.94286,1.00000,0.95798,0.85038,0.74331,0.67794,0.61105,0.56699,...,-0.31119,-0.39633,-0.71168,0.51234,-0.00231,0.18403,-0.30412,-0.50266,-0.53160,-0.48108
psd_size_at_d50_mm,0.62035,0.77370,0.83413,0.95798,1.00000,0.94748,0.86112,0.80152,0.73033,0.68340,...,-0.32534,-0.41609,-0.79644,0.61857,0.03855,0.13824,-0.27497,-0.45152,-0.47373,-0.36243
psd_size_at_d60_mm,0.55870,0.66269,0.71345,0.85038,0.94748,1.00000,0.96761,0.91539,0.83489,0.77512,...,-0.34322,-0.44345,-0.85708,0.72127,0.08921,0.10845,-0.25296,-0.32335,-0.33199,-0.33130
psd_size_at_d70_mm,0.49860,0.56438,0.60496,0.74331,0.86112,0.96761,1.00000,0.97622,0.89995,0.83078,...,-0.35537,-0.45894,-0.88402,0.78837,0.11809,0.08638,-0.23687,-0.10001,-0.15904,-0.31837
psd_size_at_d80_mm,0.49457,0.51681,0.54407,0.67794,0.80152,0.91539,0.97622,1.00000,0.96088,0.90094,...,-0.37718,-0.48196,-0.91355,0.83525,0.11272,0.07089,-0.22117,-0.04578,-0.11284,-0.31544
psd_size_at_d90_mm,0.51030,0.47931,0.48903,0.61105,0.73033,0.83489,0.89995,0.96088,1.00000,0.97992,...,-0.40849,-0.50369,-0.91544,0.83355,0.09628,0.01806,-0.17254,-0.17987,-0.24064,-0.31992
psd_size_at_d95_mm,0.49970,0.44448,0.44694,0.56699,0.68340,0.77512,0.83078,0.90094,0.97992,1.00000,...,-0.41203,-0.50386,-0.88277,0.79272,0.07354,0.00346,-0.13286,-0.25077,-0.32654,-0.30671


In [7]:
# # Heatmap

# plt.figure(figsize=(12, 10))

# sns.heatmap(
#     corr_mat,
#     cmap='coolwarm',
#     annot=True,      # Show correlation values
#     fmt='.2f',       # 2 decimal places
#     vmin=-1,
#     vmax=1,
#     square=True
# )

# plt.title("Correlation Matrix")
# plt.tight_layout()
# plt.show()

In [10]:
# Checking for correlation value greater than 0.6

temp = ['psd_has_sedimentation', 'psd_passing_at_0_002mm_pct', 'psd_passing_at_0_063mm_pct',
       'psd_passing_at_2mm_pct', 
       'proctor_diam_mm', 'grain_density_g_cm3', 'hyd_cond_kf_m_s',
       'hyd_cond_hyd_gradient', 'atterberg_liquid_limit_pct',
       'atterberg_plastic_limit_pct', 'loss_on_ignition_pct']

lst = []

for col in df[temp]:
    for row in df[temp]:
        if row == col:  # want upper trianglar
            break
        elif corr_mat.loc[row, col] >= 0.6:
            lst.append([row, col])

len(lst)
# lst

8

In [11]:
# Spearman Corrlation - how strongly two variables have a monotonic relationship

# Features and targets
X = df[features]
y_mdd = df["proctor_mdd_g_cm3"]
y_owc = df["proctor_owc_pct"]

# Compute correlations
spearman_mdd = X.corrwith(y_mdd, method="spearman")
spearman_owc = X.corrwith(y_owc, method="spearman")

# Combine into one DataFrame
spearman_df = pd.DataFrame({
    "MDD": spearman_mdd,
    "OWC": spearman_owc
})

spearman_df

,MDD,OWC
psd_size_at_d10_mm,0.265255,-0.631197
psd_size_at_d20_mm,0.383545,-0.692391
psd_size_at_d30_mm,0.504238,-0.753145
psd_size_at_d40_mm,0.603969,-0.798450
psd_size_at_d50_mm,0.645085,-0.817305
psd_size_at_d60_mm,0.657909,-0.818475
psd_size_at_d70_mm,0.666471,-0.805522
psd_size_at_d80_mm,0.672704,-0.790031
psd_size_at_d90_mm,0.693887,-0.784628
psd_size_at_d95_mm,0.694491,-0.758853


In [12]:
# Mutual Information - how much knowing one variable reduces your uncertainty about another variable

mi_mdd = {}
mi_owc = {}

for feature in X.columns:
    # ----- MDD -----
    temp = pd.concat([X[feature], y_mdd], axis=1).dropna()

    mi_mdd[feature] = mutual_info_regression(
        temp[[feature]],
        temp[y_mdd.name],
        random_state=42
    )[0]

    # ----- OWC -----
    temp = pd.concat([X[feature], y_owc], axis=1).dropna()

    mi_owc[feature] = mutual_info_regression(
        temp[[feature]],
        temp[y_owc.name],
        random_state=42
    )[0]

mi_df = pd.DataFrame({
    "MDD": mi_mdd,
    "OWC": mi_owc
})

mi_df["Max MI"] = mi_df.max(axis=1)
mi_df = mi_df.sort_values("Max MI", ascending=False)

mi_df

,MDD,OWC,Max MI
psd_size_at_d50_mm,0.539305,0.702167,0.702167
psd_size_at_d60_mm,0.541154,0.657052,0.657052
psd_size_at_d80_mm,0.475530,0.652114,0.652114
psd_size_at_d40_mm,0.470169,0.618400,0.618400
psd_size_at_d70_mm,0.617714,0.589566,0.617714
atterberg_plastic_limit_pct,0.597439,0.468811,0.597439
psd_size_at_d90_mm,0.499757,0.593391,0.593391
psd_passing_at_2mm_pct,0.569822,0.565661,0.569822
psd_passing_at_0_063mm_pct,0.463335,0.560923,0.560923
psd_size_at_d95_mm,0.447011,0.524211,0.524211
